# Older people projected prevalence

The analysis below attempts to replicate the methodology used by poppi to provide population projections

In [2]:
import dotenv
import os
import fsspec
import requests
import polars as pl
import polars.selectors as cs
from pathlib import Path
from zipfile import ZipFile
import plotly.express as px
from io import BytesIO

# set the OUTPUT_DIRECTORY using an environment variable, a .env file or leave it as the current directory
dotenv.load_dotenv()
OUTPUT_DIRECTORY = Path(os.environ.get("OUTPUT_DIRECTORY", "."))

# Population Projections

I have used the 2022 based migration category variant dataset for estimating population at District level

This is in line with ONS guidance and projects populations in line with the most likely migration levels

In [3]:
response = requests.get(
    "https://www.ons.gov.uk/file?uri=/peoplepopulationandcommunity/populationandmigration/populationprojections/datasets/"
    "localauthoritiesinenglandz1/2022basedmigrationcategoryvariant/2022snpppopulationsyoamigcat.zip"
)
with ZipFile(BytesIO(response.content)) as zf:
    with zf.open("2022 SNPP Population persons.csv") as file:
        population_projections = pl.read_csv(file)
population_projections

AREA_CODE,AREA_NAME,COMPONENT,SEX,AGE_GROUP,2022,2023,2024,2025,2026,2027,2028,2029,2030,2031,2032,2033,2034,2035,2036,2037,2038,2039,2040,2041,2042,2043,2044,2045,2046,2047
str,str,str,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""E06000047""","""County Durham""","""Population""","""persons""","""0""",4642.0,4555.07,4601.733,4637.541,4637.684,4610.47,4612.793,4614.691,4617.59,4620.734,4624.431,4633.373,4642.656,4651.514,4661.897,4673.33,4685.507,4695.915,4702.826,4706.168,4706.777,4702.146,4691.745,4677.61,4660.477,4641.389
"""E06000047""","""County Durham""","""Population""","""persons""","""1""",4688.0,4697.672,4596.411,4632.471,4664.288,4663.669,4637.073,4638.906,4640.568,4642.815,4645.421,4648.552,4656.971,4666.075,4674.983,4685.527,4697.237,4709.799,4720.78,4728.332,4732.35,4733.642,4729.662,4719.802,4706.172,4689.456
"""E06000047""","""County Durham""","""Population""","""persons""","""2""",4875.0,4803.54,4807.672,4679.24,4708.729,4739.019,4738.929,4713.138,4714.156,4715.218,4716.912,4719.037,4721.679,4729.682,4738.633,4747.608,4758.292,4770.242,4783.183,4794.633,4802.711,4807.237,4809.067,4805.534,4795.99,4782.625
"""E06000047""","""County Durham""","""Population""","""persons""","""3""",5007.0,4950.82,4871.335,4861.949,4722.593,4749.6,4780.163,4781.446,4755.971,4755.884,4756.456,4757.693,4759.409,4761.66,4769.269,4778.1,4787.095,4797.915,4810.029,4823.246,4835.075,4843.581,4848.588,4850.858,4847.737,4838.532
"""E06000047""","""County Durham""","""Population""","""persons""","""4""",5121.0,5082.646,5016.72,4921.841,4910.021,4764.06,4790.946,4822.43,4823.992,4798.202,4797.165,4797.318,4798.143,4799.523,4801.412,4808.705,4817.436,4826.511,4837.43,4849.735,4863.202,4875.405,4884.307,4889.729,4892.419,4889.667
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""E07000189""","""South Somerset""","""Population""","""persons""","""87""",738.0,750.603,786.348,819.712,831.599,844.507,806.856,940.276,1068.225,1089.298,1129.705,1168.742,1496.579,1437.354,1330.853,1265.814,1278.581,1256.37,1274.98,1286.838,1265.414,1285.638,1361.155,1423.155,1490.696,1536.651
"""E07000189""","""South Somerset""","""Population""","""persons""","""88""",677.0,659.743,673.394,704.691,736.111,748.036,760.195,727.057,847.505,964.254,986.124,1023.602,1060.17,1359.683,1305.575,1210.199,1152.706,1164.027,1145.001,1163.128,1175.611,1157.415,1177.527,1247.846,1305.933,1368.545
"""E07000189""","""South Somerset""","""Population""","""persons""","""89""",516.0,604.875,593.996,608.278,635.717,664.629,675.735,687.896,659.759,766.747,873.335,895.533,932.173,965.114,1236.514,1192.612,1106.296,1055.068,1064.035,1048.165,1065.43,1078.897,1063.233,1082.142,1148.334,1202.798


In [4]:
SURREY_EAST = (
    "E07000207",  # "Elmbridge",
    "E07000208",  # "Epsom and Ewell",
    "E07000210",  # "Mole Valley",
    "E07000211",  # "Reigate and Banstead",
    "E07000215",  # "Tandridge",
)
SURREY_WEST = (
    "E07000209",  # "Guildford",
    "E07000212",  # "Runnymede",
    "E07000213",  # "Spelthorne",
    "E07000214",  # "Surrey Heath",
    "E07000216",  # "Waverley",
    "E07000217",  # "Woking",
)

lgr_mapping = population_projections.select(
    "AREA_CODE", 
    "AREA_NAME",
    LGR_ALLOCATION=pl.when(pl.col("AREA_CODE").is_in(SURREY_EAST))
    .then(pl.lit("EAST"))
    .when(pl.col("AREA_CODE").is_in(SURREY_WEST))
    .then(pl.lit("WEST"))
).filter(pl.col("LGR_ALLOCATION").is_not_null()).unique()
lgr_mapping

AREA_CODE,AREA_NAME,LGR_ALLOCATION
str,str,str
"""E07000213""","""Spelthorne""","""WEST"""
"""E07000212""","""Runnymede""","""WEST"""
"""E07000209""","""Guildford""","""WEST"""
"""E07000210""","""Mole Valley""","""EAST"""
"""E07000208""","""Epsom and Ewell""","""EAST"""
…,…,…
"""E07000211""","""Reigate and Banstead""","""EAST"""
"""E07000217""","""Woking""","""WEST"""
"""E07000214""","""Surrey Heath""","""WEST"""


In [5]:
older_population = population_projections.with_columns(
    pl.col("AGE_GROUP").str.extract(r"\d+", 0).cast(pl.Int32)
).join(
    lgr_mapping,
    on=["AREA_CODE", "AREA_NAME"]
).filter(
    pl.col("AGE_GROUP") >= 65
)
older_population

AREA_CODE,AREA_NAME,COMPONENT,SEX,AGE_GROUP,2022,2023,2024,2025,2026,2027,2028,2029,2030,2031,2032,2033,2034,2035,2036,2037,2038,2039,2040,2041,2042,2043,2044,2045,2046,2047,LGR_ALLOCATION
str,str,str,str,i32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str
"""E07000207""","""Elmbridge""","""Population""","""persons""",65,1325.0,1405.216,1446.266,1496.794,1606.548,1641.406,1627.566,1593.77,1711.092,1652.082,1638.329,1707.634,1667.415,1703.073,1728.167,1707.027,1776.545,1694.135,1733.253,1695.493,1709.601,1720.972,1774.315,1835.478,1750.418,1758.979,"""EAST"""
"""E07000207""","""Elmbridge""","""Population""","""persons""",66,1239.0,1303.714,1381.642,1421.37,1469.71,1575.258,1610.508,1598.877,1567.673,1680.088,1624.109,1610.764,1677.164,1638.512,1672.112,1697.199,1676.432,1742.18,1662.695,1700.36,1663.658,1677.091,1688.626,1741.673,1801.854,1720.328,"""EAST"""
"""E07000207""","""Elmbridge""","""Population""","""persons""",67,1249.0,1219.538,1282.195,1357.265,1396.542,1444.385,1545.739,1581.26,1571.875,1542.546,1650.465,1597.511,1584.107,1648.404,1610.92,1642.712,1667.833,1646.812,1709.564,1632.651,1669.118,1633.4,1646.149,1657.773,1710.677,1770.336,"""EAST"""
"""E07000207""","""Elmbridge""","""Population""","""persons""",68,1223.0,1233.501,1205.3,1264.301,1337.322,1377.519,1424.554,1523.006,1558.562,1550.962,1523.503,1627.718,1577.27,1563.93,1626.374,1589.961,1620.253,1645.499,1624.415,1684.598,1609.821,1645.277,1610.416,1622.616,1634.446,1687.345,"""EAST"""
"""E07000207""","""Elmbridge""","""Population""","""persons""",69,1240.0,1208.031,1218.864,1191.978,1249.865,1320.843,1361.034,1407.603,1503.958,1539.538,1533.222,1507.51,1608.746,1560.458,1547.299,1607.935,1572.55,1601.652,1626.708,1606.092,1663.804,1590.914,1625.241,1591.09,1602.838,1614.683,"""EAST"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""E07000217""","""Woking""","""Population""","""persons""",86,391.0,394.444,391.308,388.11,377.187,350.555,402.522,425.813,461.057,463.591,486.177,581.486,582.922,537.055,529.0,504.828,496.827,516.224,493.019,514.232,512.892,548.243,574.98,566.229,592.175,601.778,"""WEST"""
"""E07000217""","""Woking""","""Population""","""persons""",87,303.0,363.398,366.612,364.805,362.484,352.233,329.011,376.378,399.167,432.976,436.055,457.91,547.221,550.211,507.293,499.146,476.785,469.465,487.789,467.383,487.136,486.64,519.919,545.433,538.829,563.427,"""WEST"""
"""E07000217""","""Woking""","""Population""","""persons""",88,295.0,276.973,333.178,336.067,335.717,334.818,325.309,305.036,347.297,368.897,401.271,405.02,425.86,507.639,513.185,473.454,465.49,444.613,438.052,455.416,437.629,455.761,455.999,487.01,510.946,506.366,"""WEST"""


# Indicator 1 - Dementia
I chose to sourc

In [6]:
dementia = pl.read_csv("https://fingertips.phe.org.uk/api/all_data/csv/for_one_indicator?indicator_id=92949").filter(
    pl.col("Area Code").is_in(SURREY_EAST + SURREY_WEST),
)
dementia

Indicator ID,Indicator Name,Parent Code,Parent Name,Area Code,Area Name,Area Type,Sex,Age,Category Type,Category,Time period,Value,Lower CI 95.0 limit,Upper CI 95.0 limit,Lower CI 99.8 limit,Upper CI 99.8 limit,Count,Denominator,Value note,Recent Trend,Compared to England value or percentiles,Column not used,Time period Sortable,New data,Compared to goal,Time period range
i64,str,str,str,str,str,str,str,str,str,str,i64,f64,f64,f64,str,str,i64,f64,str,str,str,str,i64,str,str,str
92949,"""Estimated dementia diagnosis r…","""E92000001""","""England""","""E07000207""","""Elmbridge""","""District""","""Persons""","""65+ yrs""",null,null,2017,64.2,57.0,70.8,null,null,1142,1777.6,null,null,"""Similar""","""Not compared""",20170000,null,"""Amber""","""1y"""
92949,"""Estimated dementia diagnosis r…","""E92000001""","""England""","""E07000208""","""Epsom and Ewell""","""District""","""Persons""","""65+ yrs""",null,null,2017,59.0,51.5,65.8,null,null,591,1001.6,null,null,"""Worse""","""Not compared""",20170000,null,"""Red""","""1y"""
92949,"""Estimated dementia diagnosis r…","""E92000001""","""England""","""E07000209""","""Guildford""","""District""","""Persons""","""65+ yrs""",null,null,2017,60.3,53.0,66.9,null,null,770,1276.8,null,null,"""Worse""","""Not compared""",20170000,null,"""Amber""","""1y"""
92949,"""Estimated dementia diagnosis r…","""E92000001""","""England""","""E07000210""","""Mole Valley""","""District""","""Persons""","""65+ yrs""",null,null,2017,64.6,57.0,71.4,null,null,891,1379.0,null,null,"""Similar""","""Not compared""",20170000,null,"""Amber""","""1y"""
92949,"""Estimated dementia diagnosis r…","""E92000001""","""England""","""E07000211""","""Reigate and Banstead""","""District""","""Persons""","""65+ yrs""",null,null,2017,69.7,61.8,76.7,null,null,1228,1762.3,null,null,"""Similar""","""Not compared""",20170000,null,"""Amber""","""1y"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
92949,"""Estimated dementia diagnosis r…","""E92000001""","""England""","""E07000213""","""Spelthorne""","""District""","""Persons""","""65+ yrs""",null,null,2025,65.2,57.6,72.1,null,null,893,1369.5,null,"""No significant change""","""Similar""","""Not compared""",20250000,null,"""Amber""","""1y"""
92949,"""Estimated dementia diagnosis r…","""E92000001""","""England""","""E07000214""","""Surrey Heath""","""District""","""Persons""","""65+ yrs""",null,null,2025,69.7,61.7,76.8,null,null,1094,1569.8,null,"""No significant change""","""Similar""","""Not compared""",20250000,null,"""Amber""","""1y"""
92949,"""Estimated dementia diagnosis r…","""E92000001""","""England""","""E07000215""","""Tandridge""","""District""","""Persons""","""65+ yrs""",null,null,2025,71.2,63.2,78.3,null,null,1078,1513.1,null,"""No significant change""","""Similar""","""Not compared""",20250000,null,"""Amber""","""1y"""


In [7]:
years_to_check = dementia["Time period"].unique().cast(pl.String).to_list()
older_population.group_by(
    "AREA_CODE", "AREA_NAME"
).agg(
    cs.by_name(years_to_check, require_all=False).sum()
)

AREA_CODE,AREA_NAME,2022,2023,2024,2025
str,str,f64,f64,f64,f64
"""E07000216""","""Waverley""",28757.0,29235.599,29800.165,30341.595
"""E07000213""","""Spelthorne""",18666.0,18768.044,18974.214,19138.92
"""E07000207""","""Elmbridge""",25357.0,25730.187,26170.144,26640.989
"""E07000217""","""Woking""",17592.0,17847.576,18054.206,18303.44
"""E07000209""","""Guildford""",25577.0,25888.657,26245.866,26614.089
…,…,…,…,…,…
"""E07000210""","""Mole Valley""",21237.0,21480.702,21832.47,22171.527
"""E07000212""","""Runnymede""",15333.0,15524.244,15786.168,16045.548
"""E07000215""","""Tandridge""",18722.0,18943.853,19218.085,19486.364


In [14]:
dementia_pop_by_year = older_population.group_by(
    "AREA_CODE", "AREA_NAME", "LGR_ALLOCATION"
).agg(
    cs.by_name(years_to_check, require_all=False).sum()
).unpivot(
    cs.by_name(years_to_check, require_all=False),
    index=["AREA_CODE", "AREA_NAME", "LGR_ALLOCATION"],
    variable_name="year",
    value_name="estimated_population"
).with_columns(
    pl.col("year")
)
dementia_pop_by_year

AREA_CODE,AREA_NAME,LGR_ALLOCATION,year,estimated_population
str,str,str,str,f64
"""E07000215""","""Tandridge""","""EAST""","""2022""",18722.0
"""E07000213""","""Spelthorne""","""WEST""","""2022""",18666.0
"""E07000208""","""Epsom and Ewell""","""EAST""","""2022""",14847.0
"""E07000212""","""Runnymede""","""WEST""","""2022""",15333.0
"""E07000214""","""Surrey Heath""","""WEST""","""2022""",18377.0
…,…,…,…,…
"""E07000216""","""Waverley""","""WEST""","""2025""",30341.595
"""E07000207""","""Elmbridge""","""EAST""","""2025""",26640.989
"""E07000210""","""Mole Valley""","""EAST""","""2025""",22171.527


In [15]:
dementia_vis = dementia.select(
    AREA_CODE="Area Code",
    estimated_count="Denominator",
    year=pl.col("Time period").cast(pl.String)
).join(
    dementia_pop_by_year,
    on=["AREA_CODE", "year"]
).with_columns(
    estimated_prevalence=(pl.col("estimated_count") / pl.col("estimated_population")).round(3) * 100
)
dementia_vis

AREA_CODE,estimated_count,year,AREA_NAME,LGR_ALLOCATION,estimated_population,estimated_prevalence
str,f64,str,str,str,f64,f64
"""E07000207""",1874.6,"""2022""","""Elmbridge""","""EAST""",25357.0,7.4
"""E07000208""",1050.4,"""2022""","""Epsom and Ewell""","""EAST""",14847.0,7.1
"""E07000209""",1388.5,"""2022""","""Guildford""","""WEST""",25577.0,5.4
"""E07000210""",1528.5,"""2022""","""Mole Valley""","""EAST""",21237.0,7.2
"""E07000211""",1851.2,"""2022""","""Reigate and Banstead""","""EAST""",27529.0,6.7
…,…,…,…,…,…,…
"""E07000213""",1369.5,"""2025""","""Spelthorne""","""WEST""",19138.92,7.2
"""E07000214""",1569.8,"""2025""","""Surrey Heath""","""WEST""",19207.879,8.2
"""E07000215""",1513.1,"""2025""","""Tandridge""","""EAST""",19486.364,7.8


In [16]:
fig = px.line(
    dementia_vis,
    x="year",
    y="estimated_prevalence",
    color="AREA_NAME"
)
# brief analysis of the time series below doesn't give enough evidence to suggest prevalence changes significantly enough to warant a more rigorous approach
# a drop in reigate and banstead and increase in tandridge look to me like a change in methodology / allocation
# given it's nhs data, it's likely the estimates are based on gp registrations and this could suggest a gp that was previously allocated to reigate is now allocated to tandridge
# given reigate and tandridge are neighboring I think this is the simplest explanation

# as a result, I'll just use the latest prevalence estimate as planned
fig.show()

In [17]:
dementia_projections = older_population.group_by(
    "AREA_CODE", "AREA_NAME", "LGR_ALLOCATION"
).agg(
    projected_population_2025=pl.sum("2025"),
    projected_population_2040=pl.sum("2040")
)
dementia_projections

AREA_CODE,AREA_NAME,LGR_ALLOCATION,projected_population_2025,projected_population_2040
str,str,str,f64,f64
"""E07000217""","""Woking""","""WEST""",18303.44,22000.482
"""E07000213""","""Spelthorne""","""WEST""",19138.92,22975.208
"""E07000209""","""Guildford""","""WEST""",26614.089,32829.135
"""E07000212""","""Runnymede""","""WEST""",16045.548,19951.584
"""E07000215""","""Tandridge""","""EAST""",19486.364,23845.341
…,…,…,…,…
"""E07000210""","""Mole Valley""","""EAST""",22171.527,27576.316
"""E07000207""","""Elmbridge""","""EAST""",26640.989,34625.482
"""E07000211""","""Reigate and Banstead""","""EAST""",29017.832,36989.192


In [12]:
calculated_prevalence_in_older_people=pl.col("estimated_count_2025") / pl.col("projected_population_2025")

dementia_district = dementia.filter(
    # extract latest estimates for 2025 only
    pl.col("Time period") == 2025
).select(
    AREA_CODE="Area Code",
    diagnosed_count_2025="Count",
    estimated_count_2025="Denominator"
).join(
    dementia_projections, on="AREA_CODE"
).select(
    "AREA_CODE",
    "AREA_NAME",
    "LGR_ALLOCATION",
    count="estimated_count_2025",
    reference_year=pl.lit(2025),
    prevalence_estimate=(calculated_prevalence_in_older_people * 100).round(1),
    estimated_count_2025="estimated_count_2025",
    projected_population_2025="projected_population_2025",
    projected_population_2040="projected_population_2040",
    projected_count_2040=pl.col("projected_population_2040") * calculated_prevalence_in_older_people
)
dementia_district

AREA_CODE,AREA_NAME,LGR_ALLOCATION,count,reference_year,prevalence_estimate,estimated_count_2025,projected_population_2025,projected_population_2040,projected_count_2040
str,str,str,f64,i32,f64,f64,f64,f64,f64
"""E07000213""","""Spelthorne""","""WEST""",1369.5,2025,7.2,1369.5,19138.92,22975.208,1644.00851
"""E07000212""","""Runnymede""","""WEST""",1061.8,2025,6.6,1061.8,16045.548,19951.584,1320.27849
"""E07000209""","""Guildford""","""WEST""",1498.6,2025,5.6,1498.6,26614.089,32829.135,1848.560051
"""E07000210""","""Mole Valley""","""EAST""",1622.4,2025,7.3,1622.4,22171.527,27576.316,2017.895072
"""E07000208""","""Epsom and Ewell""","""EAST""",1090.0,2025,7.1,1090.0,15258.447,17981.524,1284.525297
…,…,…,…,…,…,…,…,…,…
"""E07000211""","""Reigate and Banstead""","""EAST""",1758.0,2025,6.1,1758.0,29017.832,36989.192,2240.932387
"""E07000217""","""Woking""","""WEST""",1457.4,2025,8.0,1457.4,18303.44,22000.482,1751.774665
"""E07000214""","""Surrey Heath""","""WEST""",1569.8,2025,8.2,1569.8,19207.879,23945.725,1957.009366


In [51]:
def as_expr(val) -> pl.Expr:
    if isinstance(val, pl.Expr):
        return val
    if isinstance(val, str):
        return pl.col(val)
    return pl.lit(val)

def round_to_int(value, nearest: int = 1) -> pl.Expr:
    return (
        (as_expr(value) / nearest).round(mode="half_away_from_zero").cast(pl.Int32) * nearest
    )

def as_percent(numerator, denominator, decimals: int=1) -> pl.Expr:
    return (as_expr(numerator) / as_expr(denominator) * 100).round(decimals, mode="half_away_from_zero")



In [52]:
estimated_prevalence = pl.col("projected_count_2040") / pl.col("projected_population_2040")

dementia_counts = dementia_district.group_by(
    "LGR_ALLOCATION",
).agg(
    round_to_int(
        pl.sum("estimated_count_2025", "projected_population_2025", "projected_population_2040", "projected_count_2040"),
        100
    ),
).with_columns(
    estimated_increase=pl.col("projected_count_2040") - pl.col("estimated_count_2025")
)
dementia_counts

LGR_ALLOCATION,estimated_count_2025,projected_population_2025,projected_population_2040,projected_count_2040,estimated_increase
str,i32,i32,i32,i32,i32
"""WEST""",9500,129700,160500,11800,2300
"""EAST""",8000,112600,141000,10000,2000


# Sanity check
- Population of over 65's in EAST is expected to increase by just under 30000
- a prevalence of between 5 and 10 percent is somewhere between 1 in 10 and 1 in 20. Let's say 1 in 15.
- 1/15 == 2/30 == 2000/30000
- the count of people affected is expected to increase by 2000 

consider these results to have a very generous margin for error as:
- they don't account for different migration scenarios
- they barely account for projected life expectancy
- they don't account for changes in prevalence
- they barely account for age stratification (age band is 65 and over)


# Frailty indicator

A reasonable indication of frailty at local authority level appears to be the number of emergency hospital admissions for falls.